# 01b · SPUD playlist relationships
SPUD supplies Last.fm playlists with Spotify IDs. Read its SQLite database in read-only mode, audit its schema, deduplicate edges, remove orphan endpoints and recompute playlist sizes.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/data_pipeline.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
import pandas as pd
from scripts.data_pipeline import OUT, REPORTS


In [2]:
from scripts.data_pipeline import playlists
quality = playlists()

{
  "source_tracks": 737362,
  "clean_unique_spotify_tracks": 737362,
  "source_edges": 1024711,
  "clean_edges": 1024711,
  "removed_duplicate_or_invalid_edges": 0,
  "playlists": 19225,
  "duration_repairs": 11,
  "evaluation_playlists": 18409
}


In [3]:
import json
schema = json.loads((REPORTS / "spud_schema.json").read_text())
display(pd.DataFrame([{"table": table, **column} for table, columns in schema.items() for column in columns]))

,table,position,name,type,not_null,default,primary_key
0,artists,0,artistid,INTEGER,1,None,1
1,artists,1,name,TEXT,1,None,0
2,artists,2,spotifyid,TEXT,0,None,0
3,sqlite_sequence,0,name,,0,None,0
4,sqlite_sequence,1,seq,,0,None,0
5,albums,0,albumid,INTEGER,1,None,1
6,albums,1,name,TEXT,1,None,0
7,albums,2,artist,INTEGER,0,None,0
8,albums,3,spotifyid,TEXT,0,None,0
9,tracks,0,trackid,INTEGER,1,None,1


In [4]:
display(pd.read_parquet(OUT / "playlists.parquet").head(10))
display(pd.read_parquet(OUT / "playlist_edges.parquet").head(10))

,playlist_id,playlist_title,track_count,eligible_for_evaluation
0,1,Songs About Prostitutes,11,True
1,2,Post-punk,20,True
2,3,The Best of: I Fight Dragons,10,True
3,4,Evanescence,14,True
4,5,The Best of: POLYSICS,15,True
5,6,Gothic Rock,21,True
6,7,Untitled,32,True
7,8,Playlist???,27,True
8,9,A Crash Course In Music (Unfinished),66,True
9,10,The Best of: Weezer,38,True


,playlist_id,spotify_track_id
0,1,02JnoHSIDbpVW40uipjKcL
1,1,01HNAQL86oZsKECUfJiAwk
2,1,0eCxvvcJUgMmFOEv0tphgh
3,1,1pT9RHD2v3aHqENfVaFPw4
4,1,68PVZq98OxgeWBWbskYQLt
5,1,17T3ACYXS8kFxLgvlivNcc
6,1,31mt8T78x5xPfv8rz8CreU
7,1,0XJqss1HoEX3eCuDVESkWs
8,1,01HARyUMkuEtv11dEPC3u4
9,1,02v7jB3VzuHHQ1Di7eO5Q2


Playlist absence means no observed membership. We do not create fake playlists for catalog artists. Playlists with at least five distinct tracks support train/validation/test evaluation; smaller playlists supply training context only.